# Zone Finder & Plotter
Replaces the original `zones_finder_plotter.ipynb` with a clean, fast, parameterised version.
Set your parameters in **Cell 2** and run all cells.

In [ ]:
# ── 1. Setup ─────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook'

from data_fetch.data_fetch import fetch_ohlcv, get_watchlist, fetch_batch
from algos.pivot_engine  import extrems, detect_trend, compute_atr
from algos.zones         import buy_zone, sell_zone
from algos.confluence    import confluence_score
from algos.patterns      import detect_flag
print('✅ Imports OK')

In [ ]:
# ── 2. Parameters ─────────────────────────────────────────────────────────────
TICKER      = 'RELIANCE.NS'   # single stock to inspect
INTERVAL    = '1d'            # '1d' | '1wk' | '1h' | '4h' | '15m'
DAYS        = 500             # lookback days
ORDER       = 5               # pivot order (look-left/right bars)
DIRECTION   = 'buy'           # 'buy' or 'sell'
MULTIPLIER  = 1.0             # ATR multiplier for zone thresholds
MIN_STRENGTH_ATR = 0.3        # min pivot amplitude in ATR units (0 = off)
SHOW_PIVOTS = True
SHOW_ZONES  = True

In [ ]:
# ── 3. Fetch data ─────────────────────────────────────────────────────────────
df = fetch_ohlcv(TICKER, interval=INTERVAL, days=DAYS)
print(f'{TICKER} | {INTERVAL} | {len(df)} bars | {df.index[0].date()} → {df.index[-1].date()}')
df.tail(3)

In [ ]:
# ── 4. Pivot detection ────────────────────────────────────────────────────────
pivots = extrems(df, order=ORDER, min_strength_atr_mult=MIN_STRENGTH_ATR)
trend  = detect_trend(pivots)
print(f'Trend  : {trend}')
print(f'Pivots : {len(pivots)}')
print('Last 5 pivots:')
for idx, val, typ in pivots[-5:]:
    print(f'  {df.index[idx].date()}  {typ}  {val:.2f}')

In [ ]:
# ── 5. Zone detection ─────────────────────────────────────────────────────────
if DIRECTION == 'buy':
    zones, htf_flag = buy_zone(df, 0, 'NA', multiplier=MULTIPLIER)
else:
    zones  = sell_zone(df, 0, multiplier=MULTIPLIER)
    htf_flag = 0

print(f'Zones found : {len(zones)}')
print(f'HTF flag    : {htf_flag}')
for z in zones[-5:]:
    print(f'  {z[0]} → {z[1]}')

In [ ]:
# ── 6. Confluence score ───────────────────────────────────────────────────────
score = confluence_score(df, order=ORDER, direction=DIRECTION)
print('Confluence breakdown:')
for k, v in score.items():
    print(f'  {k:<16} {v}')

In [ ]:
# ── 7. Plot ────────────────────────────────────────────────────────────────────
fig = go.Figure()

# Candlesticks
fig.add_trace(go.Candlestick(
    x=df.index, open=df['Open'], high=df['High'],
    low=df['Low'], close=df['Close'], name='Price'
))

# Pivots
if SHOW_PIVOTS and pivots:
    pivot_dates = [df.index[i] for i,_,_ in pivots]
    pivot_vals  = [v for _,v,_ in pivots]
    pivot_cols  = ['red' if t=='T' else 'lime' for _,_,t in pivots]
    fig.add_trace(go.Scatter(
        x=pivot_dates, y=pivot_vals, mode='markers+lines',
        marker=dict(color=pivot_cols, size=8, symbol='circle'),
        line=dict(color='rgba(167,139,250,0.5)', width=1),
        name='Pivots'
    ))

# Zones
if SHOW_ZONES:
    zone_col = 'rgba(34,197,94,0.15)' if DIRECTION == 'buy' else 'rgba(239,68,68,0.15)'
    line_col = '#22c55e' if DIRECTION == 'buy' else '#ef4444'
    for z in zones:
        try:
            ts_in  = pd.Timestamp(z[0])
            ts_out = pd.Timestamp(z[1])
            legin_idx = df.index.get_loc(ts_in, method='nearest')
            hi = df['High'].iloc[legin_idx]
            lo = df['Low'].iloc[legin_idx]
            # Shaded zone box
            fig.add_vrect(
                x0=ts_in, x1=ts_out,
                fillcolor=zone_col, opacity=0.7,
                layer='below', line_width=0
            )
            fig.add_hline(y=hi, line_color=line_col, line_width=0.8, line_dash='dot',
                          x0=ts_in, x1=ts_out)
            fig.add_hline(y=lo, line_color=line_col, line_width=0.8, line_dash='dot',
                          x0=ts_in, x1=ts_out)
        except Exception:
            pass

fig.update_layout(
    title=f'{TICKER} · {INTERVAL} · Trend: {trend} · Confluence: {score["score"]}',
    template='plotly_dark',
    xaxis_rangeslider_visible=False,
    height=650,
    legend=dict(x=0.01, y=0.99),
)
fig.show()